# SMIXAE toy notebook

Iterative SMIXAE development on the toy model. Converted from the marimo notebook `toy_smixae.py`.

In [11]:
import sys
from pathlib import Path

import plotly.graph_objects as go
import torch
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

# Ensure src/ is on the path (assumes the notebook is run from notebooks/)
_src = Path.cwd().parent / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from dataclasses import dataclass, field

import einops as eo
from sae_lens.saes.sae import (
    SAEMetadata,
    TrainCoefficientConfig,
    TrainingSAEConfig,
    TrainStepOutput,
)
from sae_lens.synthetic import train_toy_sae
from torch import nn
from transformer_lens.hook_points import HookPoint
from typing_extensions import override

import smixae  # noqa: F401 -- registers architecture with SAELens
from smixae.base_smixae import (
    BaseSMIXAE,
)
from toy.metrics import compute_cofiring_matrix, compute_metrics, compute_restricted_r2
from toy.plot import plot_all_experts_with_originals
from toy.zoo import (
    EvalData,
    ManifoldActivationGenerator,
    ManifoldZoo,
    build_manifold_zoo,
    generate_eval_set,
)

# Optuna is only needed for the optional hyperparameter search (RUN_HYPEROPT).
# Import it lazily so the notebook still loads without it installed.
import optuna

display(Markdown("## Imports loaded"))

from einops.layers.torch import EinMix as Mix

import smixae.einops_norms as eon
from smixae.batchtopknorm import BatchTopKNormLayerConfig
from smixae.grump_relu import GrumpReLULayerConfig
from smixae.sparsity_layer import SparsityLayer, SparsityLayerConfig

## Imports loaded

In [12]:
import plotly.io as pio

pio.renderers.default = "notebook"

In [ ]:
# -- Dataset params --------------------------------------------------
seed        = 0
d_in        = 128
l0          = 4           # active *sparse* manifolds per sample (dense are always active)
sigma_bias  = 0.0

# Dense/sparse split + off-origin shifts (Feature Norm Hypothesis).
# frac_dense of the 48 instances are dense (always-active, origin-passing, no
# shift); the rest are sparse and translated inside their subspace so their
# minimum active norm equals a per-instance lognormal floor.
frac_dense        = 0.00  # fraction of instances marked dense
norm_floor        = 0.1   # median min active norm for sparse instances
norm_floor_spread = 0.5   # lognormal spread (sigma) of the per-instance floor

# -- Model architecture -----------------------------------------------
n_experts    = 48         # should match n_instances in zoo (48)
d_expert     = 16
d_bottleneck = 3          # 3-D for direct visualisation
d_sae = n_experts * d_expert

# -- Training -----------------------------------------------------------
k_experts         = 4     # active experts per sample (BatchTopK budget)
training_samples  = 500_000_000
batch_size        = 2 ** 13
lr                = 5e-3
lr_warm_up_steps  = 2_000
n_snapshots       = 20    # how many times compute_restricted_r2 is called during training

# -- Hyperparameter search (optional) -------------------------------------
# Flip RUN_HYPEROPT to True to run an Optuna search over the GrumpReLU layer +
# sparsity schedule before the final run, maximising the mean co-firing-matched
# R2. When False, the notebook trains a single model on the baseline `hparams`
# below. Either way the downstream training + reporting cells consume `hparams`,
# so both paths behave identically apart from where the values come from.
RUN_HYPEROPT      = False
HYPEROPT_N_TRIALS = 100

# Single source of truth for the tunable hyperparameters. The Optuna search (if
# enabled) overwrites these with the best configuration it finds.
hparams = dict(
    bandwidth=2.0,
    init_threshold=0.01,
    hardness_coefficient=4.0,
    sparsity_coefficient=1.0,
    sparsity_warm_up_steps=training_samples // batch_size,  # full-length warmup
)

# -- Infrastructure -------------------------------------------------------
device       = "cuda" if torch.cuda.is_available() else "cpu"
toy_data_dir = Path("toy_data")

display(Markdown(f"""
**Config**

| Param | Value |
|---|---|
| seed | {seed} |
| d_in | {d_in} |
| l0 (sparse) | {l0} |
| sigma_bias | {sigma_bias} |
| frac_dense | {frac_dense} |
| norm_floor | {norm_floor} |
| norm_floor_spread | {norm_floor_spread} |
| n_experts | {n_experts} |
| k_experts | {k_experts} |
| training_samples | {training_samples:,} |
| run_hyperopt | {RUN_HYPEROPT} |
| device | {device} |
"""))



**Config**

| Param | Value |
|---|---|
| seed | 0 |
| d_in | 128 |
| l0 (sparse) | 4 |
| sigma_bias | 5.0 |
| frac_dense | 0.0 |
| norm_floor | 0.1 |
| norm_floor_spread | 0.5 |
| n_experts | 48 |
| k_experts | 4 |
| training_samples | 500,000,000 |
| run_hyperopt | False |
| device | cuda |


In [14]:
def _dataset_dir(base, s, d, l0_val, b, fd, nf):
    return base / f"seed{s}_d{d}_l{l0_val}_b{b:g}_fd{fd:g}_nf{nf:g}"

_ds = _dataset_dir(toy_data_dir, seed, d_in, l0_val=l0, b=sigma_bias, fd=frac_dense, nf=norm_floor)
_zoo_path  = _ds / "zoo.pt"
_eval_path = _ds / "eval.pt"

if _zoo_path.exists() and _eval_path.exists():
    zoo       = ManifoldZoo.load(_zoo_path, device=device)
    eval_data = EvalData.load(_eval_path, device="cpu")
    _source   = f"Loaded from `{_ds}`"
else:
    _ds.mkdir(parents=True, exist_ok=True)
    zoo = build_manifold_zoo(
        d_in=d_in, seed=seed, device=device, sigma_bias=sigma_bias,
        frac_dense=frac_dense, norm_floor=norm_floor, norm_floor_spread=norm_floor_spread,
    )
    zoo.save(_zoo_path)
    eval_data = generate_eval_set(zoo, n_samples=200_000, l0=l0, seed=seed + 1, device=device)
    eval_data.save(_eval_path)
    _source = f"Generated and saved to `{_ds}`"

_type_counts = {}
for _inst in zoo.instances:
    _type_counts[_inst.type_name] = _type_counts.get(_inst.type_name, 0) + 1

_n_dense = sum(_inst.is_dense for _inst in zoo.instances)

display(Markdown(f"""
**Toy Data** -- {_source}

- Zoo: {len(zoo.instances)} instances ({_n_dense} dense / {len(zoo.instances) - _n_dense} sparse), {zoo.n_atoms} atoms, d_in={zoo.d_in}
- Eval set: {eval_data.x.shape[0]:,} samples of shape {tuple(eval_data.x.shape)}
- Manifold types: {dict(sorted(_type_counts.items()))}
"""))



**Toy Data** -- Loaded from `toy_data/seed0_d128_l4_b5_fd0_nf0.1`

- Zoo: 48 instances (0 dense / 48 sparse), 120 atoms, d_in=128
- Eval set: 200,000 samples of shape (200000, 128)
- Manifold types: {'circle': 6, 'flat_disk': 6, 'helix': 6, 'mobius': 6, 'segment': 6, 'sphere': 6, 'swiss_roll': 6, 'torus': 6}


### SMIXAERebased -- self-contained copy. Edit freely; does NOT affect the library.
Base classes and weight helpers are stable library imports -- edit the classes below.

In [15]:
@dataclass
class SMIXAEV2Config(TrainingSAEConfig):
    """Configuration for training SMIXAERebasedTraining."""

    n_experts: int = 1024
    d_expert: int = 16
    d_bottleneck: int = 3
    negative_slope : float = 1e-4

    # Loss weighting lives on the SAE config (SAELens convention) so it can be
    # warmed up via get_coefficients() / CoefficientScheduler. The sparsity
    # layers themselves emit raw, unscaled losses in their loss_dict.
    sparsity_coefficient: float = 0.25
    dead_neuron_coefficient: float = 1/32
    sparsity_warm_up_steps: int = 420

    sparsity_layer_config : SparsityLayerConfig = field(default_factory=lambda: BatchTopKNormLayerConfig(
        n_neurons=1024,
        k=8,
        threshold_learning_rate=0.1,
    ))

    rescale_acts_by_decoder_norm: bool = True
    decoder_to_unit_norm : bool = False

    @override
    @classmethod
    def architecture(cls) -> str:
        """Return the SAELens architecture identifier."""
        return "smixae_rebased"

class SMIXAEV2(BaseSMIXAE):
    def __init__(self, cfg : SMIXAEV2Config):
        super().__init__(cfg)

        assert not (self.cfg.rescale_acts_by_decoder_norm and self.cfg.decoder_to_unit_norm), "Bruh"

        self.hook_sae_acts_bottleneck = HookPoint()

    def initialize_weights(self):
        dims = {
            'd_in' : self.cfg.d_in,
            'n_experts' : self.cfg.n_experts,
            'd_expert' : self.cfg.d_expert,
            'd_bottleneck' : self.cfg.d_bottleneck
        }

        # dims
        d_in = self.cfg.d_in,
        n_experts = self.cfg.n_experts,
        d_expert = self.cfg.d_expert,
        d_bottleneck = self.cfg.d_bottleneck

        factory_kwargs = {
            'dtype' : self.dtype,
            'device' : self.device
        }

        self.encoder_charts = Mix(
                        pattern='... d_in -> ... n_experts d_expert',
                        weight_shape='d_in n_experts d_expert',
                        bias_shape='n_experts d_expert',
                        d_in = self.cfg.d_in,
                        n_experts = self.cfg.n_experts,
                        d_expert = self.cfg.d_expert,
                    ).to(**factory_kwargs)

        self.leaky_relu = nn.LeakyReLU(negative_slope=self.cfg.negative_slope)

        self.encoder_bottleneck = Mix(
                        pattern='... n_experts d_expert -> ... n_experts d_bottleneck',
                        weight_shape='n_experts d_expert d_bottleneck',
                        n_experts = self.cfg.n_experts,
                        d_expert = self.cfg.d_expert,
                        d_bottleneck = self.cfg.d_bottleneck
                    ).to(**factory_kwargs)

        self.sparsity_layer = SparsityLayer.from_config(self.cfg.sparsity_layer_config)

        self.decoder = Mix(
                        pattern='... n_experts d_bottleneck -> ... d_in',
                        weight_shape='n_experts d_bottleneck d_in',
                        bias_shape='d_in',
                        d_in = self.cfg.d_in,
                        n_experts = self.cfg.n_experts,
                        d_bottleneck = self.cfg.d_bottleneck
                    ).to(**factory_kwargs)

        super().initialize_weights()

    @override
    def encode_with_hidden_pre(self, x : torch.Tensor) -> tuple[torch.Tensor, ...]:
        pre_act_chart = self.encoder_charts(x)
        h_chart = self.leaky_relu(pre_act_chart)
        pre_act_bottleneck = self.encoder_bottleneck(h_chart)
        if self.cfg.rescale_acts_by_decoder_norm:
            pre_act_bottleneck = pre_act_bottleneck * self.decoder_norm.unsqueeze(-1)
        h_bottleneck = self.sparsity_layer(pre_act_bottleneck)

        if self.cfg.rescale_acts_by_decoder_norm:
            h_bottleneck = h_bottleneck / self.decoder_norm.unsqueeze(-1)

        self.hook_sae_acts_pre(pre_act_chart)
        self.hook_sae_acts_post(h_chart)
        self.hook_sae_acts_bottleneck(h_bottleneck)

        return pre_act_chart, h_chart, pre_act_bottleneck, h_bottleneck

    @property
    def decoder_norm(self):
        effective_decoder_weights = self.decoder.weight # Not complicated since it is just one layer :)

        return eon.l2(effective_decoder_weights, 'n_experts d_bottleneck d_in -> n_experts') # Clean einops_norm
        # Just re-implements frob norm. Done in einops to clarify what we are aggregating over
        # return eo.reduce(effective_decoder_weights ** 2, "n_experts d_bottleneck d_in -> n_experts", 'sum') ** 0.5

    @override
    def encode(self, x : torch.Tensor) -> torch.Tensor:
        _, _, _, h_bottleneck = self.encode_with_hidden_pre(x)
        return h_bottleneck

    @override
    def decode(self, feature_acts: torch.Tensor) -> torch.Tensor:
        sae_out_pre = self.decoder(feature_acts)

        sae_out_pre = self.hook_sae_recons(sae_out_pre)
        sae_out_pre = self.run_time_activation_norm_fn_out(sae_out_pre)
        return self.reshape_fn_out(sae_out_pre, self.d_head)

    @override
    @torch.no_grad()
    def fold_activation_norm_scaling_factor(self, scaling_factor: float) -> None:
        """Fold activation scaling into weights and rescale threshold to match.

        Args:
            scaling_factor: The activation norm scaling factor.
        """
        # Manually apply the standard linear weight scaling (no SAELens super call).
        self.encoder_charts.weight *= scaling_factor
        self.decoder.weight /= scaling_factor
        self.decoder.bias /= scaling_factor
        self.cfg.normalize_activations = "none"

    @override
    def training_forward_pass(self, step_input):
        if self.cfg.decoder_to_unit_norm:
            with torch.no_grad():
                self.decoder.weight /= self.decoder_norm.view(-1, 1, 1)

        pre_act_chart, h_chart, pre_act_bottleneck, h_bottleneck = self.encode_with_hidden_pre(step_input.sae_in)
        sae_out = self.decode(h_bottleneck)

        h_bottleneck_norms = h_bottleneck.norm(dim=-1)

        # The sparsity layer emits raw (unscaled) losses; apply the SAE-level
        # coefficients here. "sparsity" is linearly warmed up by SAELens's
        # CoefficientScheduler; "dead_neuron" is constant (warm_up_steps=0).
        raw = self.sparsity_layer.loss_dict
        losses = {
            'mse_loss' : self.mse_loss_fn(sae_out, step_input.sae_in).sum(dim=-1).mean(),
            'sparsity_loss' : step_input.coefficients['sparsity'] * raw['sparsity_loss'],
            'dead_neuron_loss' : step_input.coefficients['dead_neuron'] * raw['dead_neuron_loss'],
        }

        total_loss = sum(losses.values())

        metrics : dict = {}

        return TrainStepOutput(
            sae_in=step_input.sae_in,
            sae_out=sae_out,
            feature_acts=eo.rearrange(pre_act_chart, 'batch_size n_experts d_expert -> batch_size (n_experts d_expert)'), # Not the actual feature acts, we put it here for compatibility purposes
            hidden_pre=eo.rearrange(pre_act_chart, 'batch_size n_experts d_bottleneck -> batch_size (n_experts d_bottleneck)'), # Same deal here
            loss=total_loss,
            losses=losses,
            metrics=metrics,
        )

    @override
    def calculate_aux_loss(self, step_input, feature_acts, hidden_pre, sae_out):
        return super().calculate_aux_loss(step_input, feature_acts, hidden_pre, sae_out)

    @override
    def get_coefficients(self) -> dict[str, TrainCoefficientConfig | float]:
        """Expose loss coefficients to the SAELens trainer.

        The sparsity coefficient is linearly warmed up from 0 to its final
        value over ``sparsity_warm_up_steps``; the dead-neuron coefficient is
        constant (``warm_up_steps=0``).

        Returns:
            Mapping of coefficient name to its warmup config.
        """
        return {
            "sparsity": TrainCoefficientConfig(
                value=self.cfg.sparsity_coefficient,
                warm_up_steps=self.cfg.sparsity_warm_up_steps,
            ),
            "dead_neuron": TrainCoefficientConfig(
                value=self.cfg.dead_neuron_coefficient,
                warm_up_steps=0,
            ),
        }


In [16]:
# -- Instantiate (mirrors _build_smixae in cli/toy.py) ------------------------

torch.manual_seed(seed)

# BatchTopKNorm gates the bottleneck: one "neuron" per expert, norm taken over
# d_bottleneck. So n_neurons == n_experts and k == k_experts (the BatchTopK budget).
batchtopk_config = BatchTopKNormLayerConfig(
    n_neurons=n_experts,
    k=k_experts,
    dead_after_n_passes=200,
    # dead_neuron_loss_coefficient=1/32,
    threshold_learning_rate=0.1,
)

model = SMIXAEV2(
    SMIXAEV2Config(
        d_in=zoo.d_in,
        n_experts=n_experts,
        d_expert=d_expert,
        d_sae=n_experts*d_expert,
        d_bottleneck=d_bottleneck,
        normalize_activations="none",
        apply_b_dec_to_input=False,
        device=device,
        sparsity_layer_config=batchtopk_config,
        metadata=SAEMetadata(model_name="synthetic_toy", hook_name="ambient"),
    )
).to(device)

_n_params = sum(p.numel() for p in model.parameters())
display(Markdown(f"**Model**: SMIXAERebasedTraining (local copy) -- {_n_params:,} parameters"))


**Model**: SMIXAERebasedTraining (local copy) -- 119,936 parameters

### Model builder + optional Optuna hyperparameter search

`build_model(hp)` constructs a fresh GrumpReLU `SMIXAEV2` from a hyperparameter
dict. When `RUN_HYPEROPT` is set (top config cell), the next cell runs an Optuna
study over the GrumpReLU layer (`bandwidth`, `init_threshold`,
`hardness_coefficient`), the `sparsity_coefficient`, and the sparsity
`warm_up_steps`, maximising the mean co-firing-matched R2 and writing the
winners back into `hparams`. The final model below is always built from
`hparams`, so both the search-on and search-off paths train and report
identically.

In [17]:
# def build_model(hp):
#     """Build a fresh GrumpReLU SMIXAEV2 from a hyperparameter dict.

#     Shared by the Optuna objective and the final model build so both paths
#     construct identical architectures and only the tunable values differ.

#     Args:
#         hp: Mapping with keys ``bandwidth``, ``init_threshold``,
#             ``hardness_coefficient``, ``sparsity_coefficient`` and
#             ``sparsity_warm_up_steps``.

#     Returns:
#         A freshly initialised SMIXAEV2 on ``device``.
#     """
#     torch.manual_seed(seed)

#     grump_relu_config = GrumpReLULayerConfig(
#         n_neurons=n_experts,
#         bandwidth=hp["bandwidth"],
#         init_threshold=hp["init_threshold"],
#         hardness_coefficient=hp["hardness_coefficient"],
#     )

#     return SMIXAEV2(
#         SMIXAEV2Config(
#             d_in=zoo.d_in,
#             n_experts=n_experts,
#             d_expert=d_expert,
#             d_sae=n_experts * d_expert,
#             d_bottleneck=d_bottleneck,
#             normalize_activations="none",
#             apply_b_dec_to_input=False,
#             rescale_acts_by_decoder_norm=False,
#             decoder_to_unit_norm=True,
#             device=device,
#             sparsity_layer_config=grump_relu_config,
#             sparsity_coefficient=hp["sparsity_coefficient"],
#             dead_neuron_coefficient=3e-6,
#             sparsity_warm_up_steps=int(hp["sparsity_warm_up_steps"]),
#             metadata=SAEMetadata(model_name="synthetic_toy", hook_name="ambient"),
#         )
#     ).to(device)


In [18]:
# # -- Optional Optuna hyperparameter search ----------------------------------
# # Each trial builds a fresh model from sampled hyperparameters, trains it on the
# # full budget, and is scored by the mean co-firing-matched R2
# # (compute_restricted_r2 matches each manifold to its strongest co-firing expert,
# # then measures how well that expert's bottleneck linearly reconstructs the
# # manifold's intrinsic coordinates). The best params overwrite `hparams`, so the
# # downstream cells train + report on the winning configuration.
# _total_steps = training_samples // batch_size  # warmup-step upper bound


# def _hyperopt_objective(trial):
#     """Train a model on sampled hyperparameters and return mean matched R2.

#     Args:
#         trial: Optuna trial supplying the hyperparameter suggestions.

#     Returns:
#         Mean co-firing-matched R2 over all manifold instances (maximised).
#     """
#     hp = {
#         "bandwidth":              trial.suggest_float("bandwidth", 0.5, 5.0),
#         "init_threshold":         trial.suggest_float("init_threshold", 1e-3, 5e-1, log=True),
#         "hardness_coefficient":   trial.suggest_float("hardness_coefficient", 4.0, 16.0, log=False),
#         "sparsity_coefficient":   trial.suggest_float("sparsity_coefficient", 0.1, 10.0, log=False),
#         "sparsity_warm_up_steps": _total_steps, #trial.suggest_int("sparsity_warm_up_steps", 0, _total_steps),
#         "learning_rate": trial.suggest_float("learning_rate", 5e-4,1e-2, log=True)
#     }

#     trial_model = build_model(hp)

#     train_toy_sae(
#         sae=trial_model,
#         feature_dict=zoo.feature_dict,
#         activations_generator=ManifoldActivationGenerator(zoo=zoo, l0=l0, device=device),
#         training_samples=training_samples,
#         batch_size=batch_size,
#         lr=hp['learning_rate'],
#         lr_warm_up_steps=lr_warm_up_steps,
#         device=device,
#         n_snapshots=1,
#         snapshot_fn=lambda _trainer: None,
#     )
#     trial_model.eval()

#     r2_trial, _, _ = compute_restricted_r2(trial_model, zoo, eval_data, device=device)
#     return float(r2_trial.mean())


# if RUN_HYPEROPT:
#     assert optuna is not None, "RUN_HYPEROPT is True but optuna is not installed (`uv add optuna` / `uv sync`)."

#     study = optuna.create_study(direction="maximize")
#     study.optimize(_hyperopt_objective, n_trials=HYPEROPT_N_TRIALS)

#     hparams.update(study.best_params)

#     _best_rows = "\n".join(f"| {k} | {v} |" for k, v in study.best_params.items())
#     display(Markdown(f"""
# **Optuna search complete** -- {HYPEROPT_N_TRIALS} trials

# Best mean co-firing-matched R2: **{study.best_value:.4f}**

# | Param | Best value |
# |---|---|
# {_best_rows}

# `hparams` updated with the best configuration above.
# """))
# else:
#     display(Markdown("**Hyperparameter search disabled** (`RUN_HYPEROPT = False`) -- using baseline `hparams`."))


In [19]:
# Final model -- built from `hparams` (baseline defaults, or the Optuna best if
# the search ran above). Training + reporting below are identical either way.
# model = build_model(hparams)

_n_params = sum(p.numel() for p in model.parameters())
display(Markdown(f"""
**Model**: SMIXAERebasedTraining (local copy) -- {_n_params:,} parameters

Built from `hparams`:

| Param | Value |
|---|---|
| bandwidth | {hparams['bandwidth']} |
| init_threshold | {hparams['init_threshold']} |
| hardness_coefficient | {hparams['hardness_coefficient']} |
| sparsity_coefficient | {hparams['sparsity_coefficient']} |
| sparsity_warm_up_steps | {int(hparams['sparsity_warm_up_steps'])} |
"""))



**Model**: SMIXAERebasedTraining (local copy) -- 119,936 parameters

Built from `hparams`:

| Param | Value |
|---|---|
| bandwidth | 2.0 |
| init_threshold | 0.01 |
| hardness_coefficient | 4.0 |
| sparsity_coefficient | 1.0 |
| sparsity_warm_up_steps | 30517 |


In [20]:
# History is collected via snapshot callbacks -- these are co-firing matched
# metrics, NOT the reconstruction R2 that SAELens reports internally.
history: dict[str, list] = {
    "sample":   [],
    "r2":       [],   # mean co-firing matched R2 across all instances
    "cofiring": [],   # mean co-firing rate of matched expert
    "mse":      [],
    "dead":     [],
}

_total_samples = training_samples

def _snapshot_fn(trainer) -> None:
    """Called n_snapshots times, evenly spaced during training."""
    _sae = trainer.sae
    _sae.eval()

    _r2, _cf, _ = compute_restricted_r2(_sae, zoo, eval_data, device=device)
    _m          = compute_metrics(_sae, zoo, eval_data, device=device)

    history["sample"].append(trainer.n_training_samples)
    history["r2"].append(float(_r2.mean()))
    history["cofiring"].append(float(_cf.mean()))
    history["mse"].append(_m["mse"])
    history["dead"].append(_m["dead_experts"])

    _sae.train()

_manifold_ag = ManifoldActivationGenerator(zoo=zoo, l0=l0, device=device)

_save_dir = toy_data_dir / "notebook_model"
_save_dir.mkdir(parents=True, exist_ok=True)

train_toy_sae(
    sae=model,
    feature_dict=zoo.feature_dict,
    activations_generator=_manifold_ag,
    training_samples=_total_samples,
    batch_size=batch_size,
    lr=lr,
    lr_warm_up_steps=lr_warm_up_steps,
    device=device,
    n_snapshots=n_snapshots,
    snapshot_fn=_snapshot_fn,
)
model.eval()

# Final evaluation
r2_final, cofiring_final, best_experts = compute_restricted_r2(model, zoo, eval_data, device=device)
metrics_final = compute_metrics(model, zoo, eval_data, device=device)
cofiring_matrix = compute_cofiring_matrix(model, zoo, eval_data, device=device)

display(Markdown(f"""
**Training complete**

| Metric | Value |
|---|---|
| Mean R2 (co-firing matched) | {r2_final.mean():.4f} |
| Mean co-firing rate | {cofiring_final.mean():.3f} |
| MSE | {metrics_final['mse']:.5f} |
| Effective L0 | {metrics_final['effective_l0']:.2f} |
| Dead experts | {metrics_final['dead_experts']} |
"""))


Training SAE:   0%|          | 0/500000000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
assert history["sample"], "No snapshots recorded -- increase n_snapshots."

_fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Mean R2 (co-firing matched)", "Mean co-firing rate",
        "MSE", "Dead experts",
    ],
)

_xs = history["sample"]

_fig.add_trace(go.Scatter(x=_xs, y=history["r2"], mode="lines+markers", name="R2"), row=1, col=1)
_fig.add_trace(go.Scatter(x=_xs, y=history["cofiring"], mode="lines+markers", name="co-fire"), row=1, col=2)
_fig.add_trace(go.Scatter(x=_xs, y=history["mse"], mode="lines+markers", name="MSE"), row=2, col=1)
_fig.add_trace(go.Scatter(x=_xs, y=history["dead"], mode="lines+markers", name="dead"), row=2, col=2)

_fig.update_xaxes(title_text="Training samples")
_fig.update_layout(title="Training curves (co-firing matched metrics)", height=600, showlegend=False)
_fig


In [ ]:
import collections as _col

_type_r2: dict[str, list] = _col.defaultdict(list)
for _i, _inst in enumerate(zoo.instances):
    _type_r2[_inst.type_name].append(float(r2_final[_i]))

_types  = sorted(_type_r2)
_means  = [sum(_type_r2[t]) / len(_type_r2[t]) for t in _types]
_mins   = [min(_type_r2[t]) for t in _types]
_maxs   = [max(_type_r2[t]) for t in _types]

_fig_bar = go.Figure([
    go.Bar(
        name="mean R2", x=_types, y=_means,
        error_y=dict(
            type="data",
            symmetric=False,
            array=[_maxs[i] - _means[i] for i in range(len(_types))],
            arrayminus=[_means[i] - _mins[i] for i in range(len(_types))],
        ),
    )
])
_fig_bar.update_layout(
    title="R2 by manifold type (mean +/- min/max across variants)",
    yaxis_title="R2",
    xaxis_title="Manifold type",
    yaxis_range=[0, 1],
)
_fig_bar


In [ ]:
_labels = [f"{inst.type_name}[{inst.variant_idx}]" for inst in zoo.instances]
_fig_heat = go.Figure(go.Heatmap(
    z=cofiring_matrix.numpy(),
    x=[str(e) for e in range(cofiring_matrix.shape[1])],
    y=_labels,
    colorscale="Viridis",
    colorbar=dict(title="P(fire | active)"),
))
_fig_heat.update_layout(
    title="Co-firing matrix: P(expert e fires | manifold i active)",
    xaxis_title="Expert index",
    yaxis_title="Manifold instance",
    height=800,
)
_fig_heat


### All 48 experts: learned (red) vs original (blue)

In [ ]:
fig = plot_all_experts_with_originals(
    model, zoo, eval_data, best_experts,
    k_experts=k_experts, r2=r2_final, cofiring=cofiring_final, device=device,
)

# fig.show()
fig.write_html('bruh.html', include_plotlyjs='cdn')